In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [22]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run(["random", "random"])

from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, COMET_SPAWN_STEPS

obs = env.steps[1][0].observation
planets = [Planet(*p) for p in obs.planets]

print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")

print(f"Initial Planet {obs.initial_planets}")

obs = env.steps[COMET_SPAWN_STEPS[0]][0].observation
comet_planet_ids = obs.get("comet_planet_ids", [])
print(f"Commets {len(comet_planet_ids)}")

Player: 0
Angular velocity: 0.0390 rad/turn

Planets (32):
Initial Planet [[0, -1, 80.02787039650892, 98.35751048843838, 1.0, 11, 1], [1, -1, 1.642489511561621, 80.02787039650892, 1.0, 11, 1], [2, -1, 98.35751048843838, 19.972129603491084, 1.0, 11, 1], [3, -1, 19.972129603491084, 1.642489511561621, 1.0, 11, 1], [4, -1, 83.22338077736208, 88.00473182396084, 2.6094379124341005, 54, 5], [5, -1, 11.995268176039161, 83.22338077736208, 2.6094379124341005, 54, 5], [6, -1, 88.00473182396084, 16.77661922263792, 2.6094379124341005, 54, 5], [7, -1, 16.77661922263792, 11.995268176039161, 2.6094379124341005, 54, 5], [8, -1, 64.5948276886074, 97.6044743850757, 2.09861228866811, 54, 3], [9, -1, 2.3955256149242956, 64.5948276886074, 2.09861228866811, 54, 3], [10, -1, 97.6044743850757, 35.405172311392604, 2.09861228866811, 54, 3], [11, -1, 35.405172311392604, 2.3955256149242956, 2.09861228866811, 54, 3], [12, -1, 70.85024329134956, 80.94743298496765, 1.6931471805599454, 20, 2], [13, -1, 19.052567015032

In [ ]:
# %%writefile submission.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet, COMET_SPAWN_STEPS

sun_config = (50.0, 50.0, 10.0)
time = 0
launch_positions = [] # launch pos values: id, x, y, radius, distance (to target), fleet travelling speed

def find_planet(planets, id):
    for p in planets:
        if p.id == id:
            return p
    return None
        
def cal_intercept(planet, angular_v, launcher_pos, time, fleet_speed):
    dx = planet.x - sun_config[0]
    dy = planet.y - sun_config[1]
    
    current_angle = math.atan2(dy, dx)
    planet_r = math.sqrt(dx**2 + dy**2)
    t = time
    
    for _ in range(20):
        target_angle = current_angle + (angular_v * t) # calculate the next angle given the time, t
        # cal position from the new angle
        tx = sun_config[0] + (planet_r * math.cos(target_angle))
        ty = sun_config[1] + (planet_r * math.sin(target_angle))
        
        # cal the distance and time the planet would travel
        dist = math.sqrt((tx - launcher_pos.x)**2 + (ty - launcher_pos.y)**2)
        travel_time = dist / fleet_speed
        
        if abs(t - travel_time) < 0.1: # Threshold for 'close enough'
            break
        t = travel_time
    
    return tx, ty

def cal_intercept_with_sun(start_x, start_y, nearest_x, nearest_y, angle):
    dx = sun_config[0] - start_x
    dy = sun_config[1] - start_y
    
    nx = sun_config[0] - nearest_x
    ny = sun_config[1] - nearest_y
    
    miss_distance = abs(dx * math.sin(angle) - dy * math.cos(angle))
    dot_product = dx * math.cos(angle) + dy * math.sin(angle)
    
    nearest_angle = math.atan2(ny, nx)
    nearest_dot_product = nx * math.cos(nearest_angle) + ny * math.sin(nearest_angle)
    return miss_distance, dot_product, nearest_dot_product

def get_prev_image(initial_targets, nearest):
    for p in initial_targets:
        if p.id == nearest.id and nearest.x != p.x: # break if we found our planet and its revolving
            return p
        elif p.id == nearest.id: # break if we found our planet
            return None
        
def get_angle(dx, dy, initial_nearest, nearest, angular_velocity, my_planet, min_dist, fleet_speed):
    angle = math.atan2(dy, dx)
    tx, ty = nearest.x, nearest.y
    if initial_nearest != None:
        tx, ty = cal_intercept(nearest, angular_velocity, my_planet, min_dist/fleet_speed, fleet_speed)
        angle = math.atan2(ty - my_planet.y, tx - my_planet.x)
    return angle, tx, ty

def get_inputs(obs):
    player = obs.get("player", 0)
    planets = [Planet(*p) for p in obs.get("planets", [])]
    initial_planets = [Planet(*p) for p in obs.get("initial_planets", [])]
    fleets = [Fleet(*f) for f in obs.get("fleets", [])]
    
    comet_planet_ids = obs.get("comet_planet_ids", [])

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    initial_targets = [p for p in initial_planets if p.owner != player]
    
    fleets_owned = [f for f in fleets if f.owner == player]
    
    return my_planets, targets, initial_targets, fleets_owned, comet_planet_ids

def cal_fleet_speed(num_of_ships):
    return 1.0 + (6.0 - 1.0) * (math.log(num_of_ships) / math.log(1000)) ** 1.5
    
def is_fleet_in_transit(angle, launch_pos, target_planet: Planet, angular_velocity, owned_fleet):
    for f in owned_fleet:
        if f.from_planet_id == launch_pos[0]:
            # orbital radius, distance from sun to planet
            fx, fy = target_planet.x, target_planet.y
            if angular_velocity > 0:
                sun_planet_radius = math.hypot(target_planet.x - sun_config[0], target_planet.y - sun_config[1])
                
                # Calculate current angle of target from the sun
                theta = math.atan2(target_planet.y - sun_config[1], target_planet.x - sun_config[0])
                
                # calculate future angle
                theta = theta + (angular_velocity * launch_pos[4] / launch_pos[5])
                
                # get new coordinates
                fx = sun_config[0] + sun_planet_radius * math.cos(theta)
                fy = sun_config[1] + sun_planet_radius * math.sin(theta)
            
            tx = launch_pos[1] + launch_pos[4] * math.cos(angle)
            ty = launch_pos[2] + launch_pos[4] * math.sin(angle)
            
            print(f"Target is at: ({fx:.2f}, {fy:.2f})")
            print(f"Calculated Point is at: ({tx:.2f}, {ty:.2f})")
            print(f"Using Angle: {math.degrees(angle):.2f} and Distance: {launch_pos[4]}")
            
            # calculate euclidean distance of the two points
            distance = math.hypot(tx - fx, ty - fy)
            print(f"Euclidean Distance: {distance}, Radius: {launch_pos[3]}, Orbiting: {True if angular_velocity > 0 else False}")
            return distance <= launch_pos[3]
    return False

def agent(obs):
    global time
    time += 1
    
    moves = []
    my_planets, targets, initial_targets, fleets_owned, comet_planet_ids = get_inputs(obs)

    if not targets:
        return []

    for mine in my_planets:
        # Find nearest 3 planets we don't own
        next_nearest_list = sorted(targets, key=lambda t: math.hypot(mine.x - t.x, mine.y - t.y))[:3]
        for nearest in next_nearest_list:
            if nearest.id in comet_planet_ids:
                continue
            
            ships_needed = nearest.ships + 1
            fleet_speed = cal_fleet_speed(ships_needed)
            if mine.ships >= ships_needed:
                initial_nearest = get_prev_image(initial_targets, nearest)
                    
                min_dist = math.sqrt((mine.x - nearest.x)**2 + (mine.y - nearest.y)**2)
                
                dx = nearest.x - mine.x
                dy = nearest.y - mine.y
                
                angular_velocity = 0
                if initial_nearest != None and (initial_nearest.x != nearest.x or initial_nearest.y != nearest.y):
                    angular_velocity = obs.angular_velocity
                    
                angle, tx, ty = get_angle(dx, dy, initial_nearest, nearest, angular_velocity, mine, min_dist, fleet_speed)
                
                should_launch = True
                for launch_pos in launch_positions:
                    if mine.id == launch_pos[0]:
                        fleet_still_alive = any(f.from_planet_id == mine.id for f in fleets_owned)
        
                        if fleet_still_alive:
                            should_launch = False 
                            break
                        else:
                            launch_positions.remove(launch_pos)
                
                if should_launch:
                    distance_r, dot_product, nearest_dot_product = cal_intercept_with_sun(mine.x, mine.y, tx, ty, angle)

                    move = [mine.id, angle, ships_needed]
                    if not (dot_product > 0 and distance_r <= sun_config[2]) or nearest_dot_product < 0:
                        moves.append(move)
                        launch_positions.append((mine.id, mine.x, mine.y, mine.radius, min_dist, fleet_speed))
    return moves

In [52]:
from kaggle_environments import make

# Test it against the random agent
env = make("orbit_wars", debug=True)
env.run([agent, "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env.render(mode="ipython", width=800, height=600)
with open("replay.html", "w", encoding="utf-8") as f:
    f.write(env.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE


In [8]:
from kaggle_environments import make

env4 = make("orbit_wars", debug=True)
env4.run([agent, "random", "random", "random"])

final = env4.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

# env4.render(mode="ipython", width=800, height=600)
with open("replay-multi.html", "w", encoding="utf-8") as f:
    f.write(env4.render(mode="html"))

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE
Player 2: reward=-1, status=DONE
Player 3: reward=-1, status=DONE
